# DAVE Documents API — Collection Routes

**Prerequisite:** run `00_auth_setup.ipynb` first.

| Method | Path | Description |
|--------|------|-------------|
| GET | `/api/collection` | List user's collections |
| GET | `/api/collection/{id}` | Get a single collection |
| GET | `/api/collection/collectioninfo/{id}` | Get document info for a collection |
| GET | `/api/collection/entities/{id}` | Get entity clusters for a collection |
| GET | `/api/collection/facetsCache/{id}` | Get facets cache for a collection |
| GET | `/api/collection/{id}/download` | Download collection as zip |
| POST | `/api/collection` | Create a new collection |
| PUT | `/api/collection/{id}` | Update a collection |
| DELETE | `/api/collection/{id}` | Delete a collection |
| POST | `/api/save` | Save annotation sets |

In [1]:
import sys, os, json, requests

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from auth_state import API_BASE, auth_headers

print(f"API base: {API_BASE}")

API base: http://localhost:3001/api


/Users/rubenagazzi/miniforge3/envs/torch/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


## GET /api/collection — List collections

In [2]:
resp = requests.get(f"{API_BASE}/collection", headers=auth_headers())
resp.raise_for_status()
collections = resp.json()
print(f"{len(collections)} collection(s) found")
for c in collections[:5]:
    print(f"  id={c.get('_id')}  name={c.get('name')}")

sample_collection_id = collections[0]["_id"] if collections else None
print(f"\nUsing sample_collection_id = {sample_collection_id}")

4 collection(s) found
  id=69d9042d3d2fd4ef4de4cc76  name=tt
  id=69d8b0a5614875952ed9c8c3  name=altro
  id=69d8b070614875952ed9c8bb  name=boh
  id=6997369eea0b820291604bca  name=Epstein

Using sample_collection_id = 69d9042d3d2fd4ef4de4cc76


## GET /api/collection/{id} — Get single collection

In [3]:
if not sample_collection_id:
    print("No collections found — skipping.")
else:
    resp = requests.get(
        f"{API_BASE}/collection/{sample_collection_id}",
        headers=auth_headers(),
    )
    resp.raise_for_status()
    coll = resp.json()
    print(json.dumps(coll, indent=2, default=str))

HTTPError: 403 Client Error: Forbidden for url: http://localhost:3001/api/collection/69d9042d3d2fd4ef4de4cc76

## GET /api/collection/collectioninfo/{id} — Document info for a collection

In [ ]:
if not sample_collection_id:
    print("No collection ID — skipping.")
else:
    resp = requests.get(
        f"{API_BASE}/collection/collectioninfo/{sample_collection_id}",
        headers=auth_headers(),
    )
    resp.raise_for_status()
    info = resp.json()
    print(f"{len(info)} document(s) in collection")
    for d in info[:3]:
        print(f"  doc_id={d.get('id') or d.get('_id')}  name={d.get('name')}")

## GET /api/collection/entities/{id} — Entity clusters

In [ ]:
if not sample_collection_id:
    print("No collection ID — skipping.")
else:
    resp = requests.get(
        f"{API_BASE}/collection/entities/{sample_collection_id}",
        headers=auth_headers(),
        # Note: this can be slow on large collections — it streams JSON
        stream=True,
    )
    resp.raise_for_status()
    clusters = resp.json()
    print(f"{len(clusters)} entity cluster(s) found")
    for cl in clusters[:3]:
        print(f"  clusterId={cl.get('clusterId')}  type={cl.get('type')}  title={cl.get('title')}")

## GET /api/collection/facetsCache/{id} — Facets cache

In [ ]:
if not sample_collection_id:
    print("No collection ID — skipping.")
else:
    resp = requests.get(
        f"{API_BASE}/collection/facetsCache/{sample_collection_id}",
        # Optional: limit children per facet group
        params={"maxChildren": 5},
        headers=auth_headers(),
    )
    resp.raise_for_status()
    facets = resp.json()
    print(f"{len(facets)} facet group(s)")
    for fg in facets[:3]:
        print(f"  key={fg.get('key')}  doc_count={fg.get('doc_count')}  children={len(fg.get('children', []))}")

## GET /api/collection/{id}/download — Download collection as zip

This endpoint streams a zip archive of all collection documents.

In [ ]:
download_collection = False  # ← set True to run (can be slow on large collections)

if download_collection and sample_collection_id:
    resp = requests.get(
        f"{API_BASE}/collection/{sample_collection_id}/download",
        headers=auth_headers(),
        stream=True,
    )
    resp.raise_for_status()
    out_path = f"/tmp/collection_{sample_collection_id}.zip"
    with open(out_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved zip to {out_path}")
else:
    print("Skipped (set download_collection=True to run).")

## POST /api/collection — Create a new collection

In [ ]:
new_collection_payload = {
    "name": "test-collection-notebook",
    "allowedUserIds": [],
    "config": {
        "typesToHide": [],
        "typesOrder": ["Person", "Location", "Organization"],
    },
}

resp = requests.post(
    f"{API_BASE}/collection",
    json=new_collection_payload,
    headers=auth_headers(),
)
resp.raise_for_status()
created_collection = resp.json()
print("Created:", json.dumps(created_collection, indent=2, default=str))

new_collection_id = created_collection.get("_id")

## PUT /api/collection/{id} — Update a collection

In [ ]:
coll_id = new_collection_id or sample_collection_id

if not coll_id:
    print("No collection ID — skipping.")
else:
    resp = requests.put(
        f"{API_BASE}/collection/{coll_id}",
        json={"name": "test-collection-notebook-updated"},
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2, default=str))

## DELETE /api/collection/{id} — Delete a collection

In [ ]:
delete_collection = False  # ← set True to delete the test collection

if delete_collection and new_collection_id:
    resp = requests.delete(
        f"{API_BASE}/collection/{new_collection_id}",
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print(resp.json())
else:
    print("Skipped (set delete_collection=True to run).")

---
## POST /api/save — Save annotation sets

Persists entity annotation sets for a document and updates the facets cache.

In [ ]:
# ── substitute real values ─────────────────────────────────────────────────────
SAVE_DOC_ID        = None   # e.g. 42 or "d038bf8c..."
SAVE_COLLECTION_ID = new_collection_id or sample_collection_id

if not SAVE_DOC_ID or not SAVE_COLLECTION_ID:
    print("Set SAVE_DOC_ID and SAVE_COLLECTION_ID to run this cell.")
else:
    save_payload = {
        "collectionId":  SAVE_COLLECTION_ID,
        "docId":         SAVE_DOC_ID,
        "annotationSets": {
            "entities_": {
                "name":        "entities_",
                "annotations": [],   # list of annotation objects
            }
        },
        "features": {},
    }

    resp = requests.post(
        f"{API_BASE}/save",
        json=save_payload,
        headers=auth_headers(),
    )
    resp.raise_for_status()
    print(json.dumps(resp.json(), indent=2, default=str))